In [ ]:
import inflation, numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation
from numpy.linalg import norm
from matplotlib import pyplot as plt
import os

In [ ]:
dFrac = 1/12
N = 12
l = 1.0 / N
d = l * dFrac
w = l - d
m, iwv, iwbv = parametric_pillows.parallelTubes(N, 1, d, w, triArea=1e-3)
visualization.plot_2d_mesh(m, pointList=iwv)

In [ ]:
def predictedContraction(N, dFrac):
    l = 1.0 / N
    d = l * dFrac
    w = l - d
    restWidth = N * w + (N - 1) * d
    contractedWidth = N * w * 2 / np.pi + (N - 1) * d
    return (restWidth, contractedWidth, restWidth / contractedWidth)

In [ ]:
predictedContraction(12, 1/10)

In [ ]:
def runExperiment(dFrac = 0, pressure = 1, outDir = None, triArea=2e-5):
    if (outDir is not None): os.makedirs(outDir, exist_ok=True)
    N = 12
    l = 1.0 / N
    d = l * dFrac
    w = l - d
    m, iwv, iwbv = parametric_pillows.parallelTubes(N, 1, d, w, triArea=triArea)
    isheet = inflation.InflatableSheet(m, iwv)

    # Inflate while holding all wall vertices in the z=0 plane
    wallZVars = []
    for vi in range(m.numVertices()):
        if (not isheet.isWallVtx(vi)): continue
        zvar = isheet.varIdx(0, vi, 2)
        if (zvar != isheet.varIdx(1, vi, 2)): raise Exception('Unfused z component')
        wallZVars.append(zvar)
    fixedVars = np.unique(wallZVars + isheet.rigidMotionPinVars)
    
    import py_newton_optimizer
    opts = py_newton_optimizer.NewtonOptimizerOptions()

    import time
    inflation.benchmark_reset()
    isheet.setUseTensionFieldEnergy(True)
    isheet.setUseHessianProjectedEnergy(False)
    opts.niter = 100
    isheet.pressure = pressure
    cr = inflation.inflation_newton(isheet, fixedVars, opts)
    
    # Align the inflated tubes (long direction) vertically
    vm = isheet.visualizationMesh()
    V = vm.vertices()
    c = np.mean(V, axis=0)
    Vcentered = V - c
    R = np.linalg.eig(Vcentered.transpose() @ Vcentered)[1]
    if (np.linalg.det(R) < 0): R[:, 2] *= -1
    vm.setVertices(Vcentered @ R @ np.array([[0, -1, 0], [1, 0, 0], [0, 0, 1]]))
    
    if (outDir is not None):
        vm.save(f'{outDir}/d{dFrac}_p{pressure}.inflated.msh')
        isheet.mesh().save(f'{outDir}/d{dFrac}_p{pressure}.uninflated.msh')
        
    # Measure the full sheet's contraction induced by the inflation
    import utils
    Vclip = vm.vertices()
    Vclip = Vclip[np.logical_and(Vclip[:, 1] > -0.2, Vclip[:, 1] < 0.2)] #trim off the tube ends before computing the bounding box
    bbClip = utils.bbox(Vclip)
    contractedLength = bbClip[1][0] - bbClip[0][0]
    bbOrig = utils.bbox(isheet.mesh().vertices())
    origLength = bbOrig[1][0] - bbOrig[0][0]
    
    contraction = origLength / contractedLength
    maxEigenvalues = [ted.eigSensitivities().Lambda()[0] for ted in isheet.triEnergyDensities()]
    
    return isheet, contraction, isheet.tensionStateHistogram(), maxEigenvalues

In [ ]:
results = {}

pressures = np.linspace(0.5, 30, 20)
dFracs = [0, 1 / 16, 1 / 10, 1 / 8]

for p in pressures:
    for dFrac in [0, 1 / 16, 1 / 10, 1 / 8]:
        results[(dFrac, p)] = list(runExperiment(dFrac, p, 'tubes'))[1:]

In [ ]:
r = runExperiment(dFracs[1], 0.5)

In [ ]:
# Check contraction behavior of zero fuse width example under mesh refinement
rHigherRes = runExperiment(dFracs[0], 0.5, triArea=1e-5)
rExtraHighRes = runExperiment(dFracs[0], 0.5, triArea=5e-6)

In [ ]:
v = Viewer(rExtraHighRes[0].visualizationMesh())
v.show()

In [ ]:
v.showWireframe()

In [ ]:
roneeigth = runExperiment(dFracs[3], 0.5)

In [ ]:
resultSheet = roneeigth[0]

In [ ]:
nt = resultSheet.mesh().numTris()
wallStretch = np.array([ted.eigSensitivities().Lambda()[0] for i, ted in enumerate(resultSheet.triEnergyDensities()) if resultSheet.isWallTri(i % nt)])
tubeStretch = np.array([ted.eigSensitivities().Lambda()[0] for i, ted in enumerate(resultSheet.triEnergyDensities()) if not resultSheet.isWallTri(i % nt)])

In [ ]:
np.median(wallStretch), np.median(tubeStretch)

In [ ]:
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.hist(wallStretch[wallStretch > 0.99], bins=100)
plt.subplot(1, 2, 2)
plt.hist(tubeStretch[tubeStretch > 0.99], bins=100)
plt.tight_layout()
plt.show()

In [ ]:
import vis
vm = resultSheet.visualizationMesh()
stretches = [ted.eigSensitivities().Lambda()[0] for ted in resultSheet.triEnergyDensities()]
stretchesField = vis.fields.ScalarField(vm, stretches, vmin=0.99, vmax=1.01)

from tri_mesh_viewer import TriMeshViewer as Viewer
iview = Viewer(vm, scalarField=stretchesField)
iview.show()

In [ ]:
[predictedContraction(10, df)[-1] for df in dFracs]

In [ ]:
[results[(df, 0.5)][0] for df in dFracs]

In [ ]:
[results[(df, pressures[10])][0] for df in dFracs]

In [ ]:
plt.plot(dFracs, [predictedContraction(10, df)[-1] for df in dFracs], label='theory')
for p in pressures[0:5]:
    plt.plot(dFracs,  [results[(df, p)][0] for df in dFracs], '.-', label=f'Pressure {p:0.3}')
plt.xlabel('Fusing wall widths')
plt.ylabel('Contraction')
plt.legend()
plt.grid()

From measurements in Rhino, we found that the under-contraction seen for low fusing gap widths was due to two effects: a slight stretching of the sheet material and a slight squashing/elongation of the tubes in the center (presumably due to pulling forces from the neighboring tubes--though it is unclear why the full sheet cannot simply contract more to relieve these forces). This squashing is lessened for the tubes at the periphery.

In [ ]:
maxEigenvalues = np.array([ted.eigSensitivities().Lambda()[0] for ted in isheet.triEnergyDensities()])
tensionStates  = np.array([ted.tensionState() for ted in isheet.triEnergyDensities()])

### Investigate the behavior of the neo-Hookean material under unixaxial and biaxial stretch experiments

In [ ]:
# For both uniaxial and biaxial stretches, the response stiffness approaches a constant slope (of 2x the material's stiffness setting) as the stretch tends to infinity.
# This slow growth can be overwhelmed by a sufficiently high pressure (since volume grows faster than surface area).

In [ ]:
tfe = inflation.IncompressibleBalloonEnergyWithHessProjection(np.array([[1, 0], [0, 1], [0, 0]]))
stretches = np.linspace(1, 5)
energies = []
tensions = []
for s in stretches:
    tfe.setF(np.array([[s, 0], [0, s], [0, 0]]))
    energies.append(tfe.energy())
    tensions.append(tfe.denergy()[0, 0])
plt.plot(stretches, tensions)
plt.grid()
plt.show()

In [ ]:
tfe = inflation.OptionalTFEJacobianBased(np.array([[1, 0], [0, 1], [0, 0]]))
# For the neo-Hookean model, the material stiffness approaches a constant slope (of 2x base stiffness).
# under a uni-axial stretch
stretches = np.linspace(1, 10, 100)
energies = []
tensions = []
for s in stretches:
    tfe.setF(np.array([[s, 0], [0, 0], [0, 0]]))
    energies.append(tfe.energy())
    tensions.append(tfe.denergy()[0, 0])
plt.plot(stretches, tensions)
plt.grid()
plt.show()